# ArchMind: Cyber-Physical IoT Electrical Monitoring Architecture
### End-to-End Engineering Notebook: Hardware Sizing, MQTT Telemetry, NestJS Backend & Real-Time Dashboards

**Author:** ArchMind Engineering Team  
**Domain:** Electrical Engineering, Embedded Systems (ESP32), Industrial IoT, Full-Stack Node.js  
**License:** Apache 2.0


## 1. System Requirements & Specifications

The objective is to design a high-reliability, cyber-physical power monitoring node capable of measuring single-phase 230V AC mains electrical parameters:
- **Mains Voltage ($V_{RMS}$):** 230V AC nominal (operating window 180V–260V AC, $50\text{ Hz}$)
- **Load Current ($I_{RMS}$):** Up to 30A AC continuous
- **Active Power ($P$):** Up to $7.2\text{ kW}$
- **Temperature Monitoring ($T$):** Substation ambient and silicon temperature ($-20^\circ\text{C}$ to $+85^\circ\text{C}$)
- **Network Transport:** MQTT over TLS 1.3 (Port 8883) with 5-second sampling cadence
- **Backend Ingestion:** Node.js / NestJS with PostgreSQL and InfluxDB persistence


## 2. Hardware Architecture & Transducer Dimensioning

### 2.1 Current Transformer Burden Resistor Sizing
We select the **SCT-013-000** non-invasive split-core current transformer:
- **Current Rating:** $30\text{ A}_{RMS}$
- **Turns Ratio ($N$):** $1:2000$
- **Peak Secondary Current ($I_{s,peak}$):**
  $$I_{s,peak} = \frac{I_{RMS} \times \sqrt{2}}{N}$$
- **ADC Target Peak Voltage ($V_{adc,peak}$):** $1.0\text{ V}$ (allowing $\pm 1.0\text{V}$ AC swing around a $1.65\text{V}$ mid-rail DC bias for a $3.3\text{V}$ ADC input range).


In [ ]:
import math

# Electrical parameters
i_rms_max = 30.0         # Amperes
turns_ratio = 2000       # SCT-013-000 ratio
v_adc_peak_target = 1.0  # Target peak voltage swing (Volts)

# Peak secondary current calculation
i_peak_primary = i_rms_max * math.sqrt(2)
i_peak_secondary = i_peak_primary / turns_ratio

# Ideal burden resistor
r_burden_ideal = v_adc_peak_target / i_peak_secondary

# Standard E24 1% resistor value selection
r_burden_selected = 47.0 # Ohms (closest standard value)

# Actual peak swing with selected resistor
v_adc_actual_peak = i_peak_secondary * r_burden_selected

# Resistor power dissipation check
i_rms_secondary = i_rms_max / turns_ratio
p_burden_dissipation = (i_rms_secondary ** 2) * r_burden_selected

print(f"Peak Secondary Current: {i_peak_secondary * 1000:.2f} mA")
print(f"Ideal Burden Resistor:   {r_burden_ideal:.2f} Ohms")
print(f"Selected Burden (1%):    {r_burden_selected:.1f} Ohms")
print(f"Actual Peak ADC Swing:   {v_adc_actual_peak:.3f} V")
print(f"Burden Resistor Diss:    {p_burden_dissipation * 1000:.3f} mW (Derating margin > 20x for 0.25W resistor)")


### 2.2 Anti-Aliasing Filter Cutoff Frequency
To prevent high-frequency inverter switching noise and harmonic aliasing from polluting our $50\text{ Hz}$ fundamental sampling, we insert an analog RC low-pass filter:
$$f_c = \frac{1}{2 \pi R C}$$


In [ ]:
r_filter = 1000.0   # 1 kOhm
c_filter = 100e-9   # 100 nF ceramic capacitor

cutoff_frequency = 1.0 / (2.0 * math.pi * r_filter * c_filter)
print(f"Anti-Aliasing Filter Cutoff Frequency (fc): {cutoff_frequency:.2f} Hz")
print("Preserves 50Hz fundamental and 3rd/5th harmonics while suppressing >10kHz switching transients.")


### 2.3 Power Supply Thermal Analysis & Derating
We compare a linear regulator (LDO) vs. an asynchronous buck regulator stepping down $12\text{V}$ auxiliary DC supply to $3.3\text{V}$ for the ESP32 ($I_{avg} = 150\text{mA}$, $I_{peak} = 380\text{mA}$ during Wi-Fi transmission bursts).


In [ ]:
vin = 12.0
vout = 3.3
i_load_peak = 0.380 # Amps
theta_ja_sot223 = 62.0 # °C/W for SOT-223 package
ambient_temp = 50.0    # Industrial enclosure ambient

# Linear Regulator Dissipation
p_ldo_loss = (vin - vout) * i_load_peak
t_j_ldo = ambient_temp + (p_ldo_loss * theta_ja_sot223)

# Switching Regulator Dissipation (Assuming 91% efficiency)
buck_efficiency = 0.91
p_out = vout * i_load_peak
p_buck_in = p_out / buck_efficiency
p_buck_loss = p_buck_in - p_out
theta_ja_tsot23 = 45.0 # For TI TPS54302
t_j_buck = ambient_temp + (p_buck_loss * theta_ja_tsot23)

print(f"LDO Power Loss:      {p_ldo_loss:.2f} W  --> Junction Temp (Tj): {t_j_ldo:.1f} °C (VIOLATION: Exceeds 125°C max!)")
print(f"Buck Regulator Loss: {p_buck_loss:.2f} W  --> Junction Temp (Tj): {t_j_buck:.1f} °C (PASS: Safe thermal margin)")


## 3. Microcontroller Selection & Watchdog Strategy

We select the **ESP32-WROOM-32E**:
- Dual-core Xtensa LX6 @ 240MHz
- Hardware cryptographic acceleration (AES, SHA-2, RSA, ECC)
- Dual-bank Flash supporting A/B partition OTA updates with automatic rollback
- Dedicated Hardware Task Watchdog Timer (`esp_task_wdt`) configured with 15-second panic threshold.


## 4. MQTT Topic & Telemetry Contract

Below is the deterministic JSON schema contract shared across firmware, backend, and dashboard.


In [ ]:
import datetime

# Example Telemetry Payload Generator adhering to schema
telemetry_payload = {
    "deviceId": "esp32-node-001",
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "voltage": 230.25,
    "current": 4.82,
    "power": round(230.25 * 4.82 * 0.98, 2),
    "temperature": 36.4,
    "frequency": 50.01,
    "powerFactor": 0.98
}

print("Sample Validated Telemetry Packet:")
print(json.dumps(telemetry_payload, indent=2))


## 5. Synthetic Telemetry Stream Simulation & Statistical Analysis

We simulate a 100-sample operational stream with Gaussian noise, voltage sags, and load transients to verify ingestion characteristics.


In [ ]:
import random

records = []
base_time = datetime.datetime(2026, 10, 8, 4, 0, 0, tzinfo=datetime.timezone.utc)

for i in range(50):
    timestamp = (base_time + datetime.timedelta(seconds=i*5)).isoformat()
    # Voltage with nominal 230V + sinusoidal variation + random ripple
    v = 230.0 + 3.0 * math.sin(i * 0.2) + random.uniform(-0.5, 0.5)
    # Current steps from 2A up to 6A
    i_val = 4.0 + 1.5 * math.cos(i * 0.15) + random.uniform(-0.1, 0.1)
    pf = 0.97 + random.uniform(-0.02, 0.02)
    p = v * i_val * pf
    temp = 35.0 + 0.1 * i + random.uniform(-0.2, 0.2)
    
    records.append({
        "timestamp": timestamp,
        "voltage": round(v, 2),
        "current": round(i_val, 2),
        "power": round(p, 2),
        "temperature": round(temp, 2),
        "powerFactor": round(pf, 3)
    })

voltages = [r["voltage"] for r in records]
currents = [r["current"] for r in records]
powers = [r["power"] for r in records]

print(f"Generated {len(records)} telemetry records.")
print(f"Voltage Window: {min(voltages):.1f}V to {max(voltages):.1f}V (Mean: {sum(voltages)/len(voltages):.1f}V)")
print(f"Current Window: {min(currents):.2f}A to {max(currents):.2f}A (Mean: {sum(currents)/len(currents):.2f}A)")
print(f"Power Window:   {min(powers):.1f}W to {max(powers):.1f}W (Mean: {sum(powers)/len(powers):.1f}W)")


## 6. Failure Mode and Effects Analysis (FMEA)

| Component / Subsystem | Failure Mode | Severity | Probability | Detection | Mitigation Architecture |
|---|---|---|---|---|---|
| **Burden Resistor** | Open circuit on CT secondary | High (Catastrophic) | Low | Sensor limit clamp | Internal TVS diode clamps secondary to 6.8V; avoids lethal inductive flyback voltage. |
| **ESP32 Wi-Fi** | AP Disconnect / DHCP timeout | Medium | High | Heartbeat timeout | Exponential backoff (1s-30s) + FreeRTOS watchdog avoids bus lockup. |
| **Mains Voltage** | Severe lightning surge (>1kV) | High | Low | MOV clamping | Littelfuse 275V Metal Oxide Varistor (MOV) + 1A fast-acting fuse. |
| **MQTT Broker** | Network partition / Broker crash | Medium | Medium | Last Will & Testament (LWT) | Broker sets status to offline; edge buffers telemetry in local ring buffer. |
| **Database Tier** | PostgreSQL container restart | Medium | Low | Connection pool retry | NestJS connection retry loop + transient in-memory ring-buffer fallback. |


## 7. Conclusion & Next Steps
The engineering calculations and data flow models verified here form the core foundation of the `ArchMind-Electrical-NodeJS-Agent` project.
For full firmware compilation, backend execution, and live dashboard setup, refer to `README.md`.
